# Spectrum And Time-Series Products

This notebook focuses on the singleton-axis products that come from
common reductions:

- `Spectrum`: shape `(1, fchans)`, collapse time.
- `TimeSeries`: shape `(tchans, 1)`, collapse frequency.

The new derived metadata records where the product came from and
what operation produced it.

In [ ]:
%matplotlib inline

from pathlib import Path

from IPython import get_ipython
from IPython.display import display
import matplotlib
_ipython = get_ipython()
if _ipython is not None:
    _ipython.run_line_magic("matplotlib", "inline")
    matplotlib.use("module://matplotlib_inline.backend_inline", force=True)

import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u

import setigen as stg

OUT = Path("generated")
OUT.mkdir(exist_ok=True)

np.set_printoptions(precision=4, suppress=True)
print("matplotlib backend:", matplotlib.get_backend())
print("setigen from:", stg.__file__)

In [ ]:
frame = stg.Frame(
    fchans=1024,
    tchans=32,
    df=1 * u.Hz,
    dt=1 * u.s,
    fch1=6e9 + 1023,
    ascending=False,
    seed=41,
    source_name="Derived product demo",
)
frame.add_noise(5, noise_type="chi2")
drift_rate = 0.6 * frame.unit_drift_rate
frame.add_constant_signal(
    f_start=frame.get_frequency(420),
    drift_rate=drift_rate,
    level=8,
    width=4 * frame.df,
    f_profile_type="gaussian",
)
frame.add_metadata({"expected_drift_rate": float(drift_rate)})

fig, ax = plt.subplots(figsize=(9, 4))
frame.plot(ftype="px", db=True, colorbar=True)
ax.set_title("Input frame")
display(fig)
plt.close(fig)

In [ ]:
dd = stg.dedrift(frame, drift_rate=drift_rate)
spectrum = dd.spectrum(mode="sum", normalize=True)
peak_index = int(np.argmax(spectrum.data))

fig, ax = plt.subplots(figsize=(10, 3))
spectrum.plot(ftype="px")
ax.axvline(peak_index, color="k", linestyle="--", alpha=0.7)
ax.set_title("Dedrifted, summed, sigma-normalized spectrum")
display(fig)
plt.close(fig)

print("peak index:", peak_index)
print("dedrift metadata:", dd.metadata["derived"])
print("spectrum metadata:", spectrum.metadata["derived"])

In [ ]:
f_center = dd.get_frequency(peak_index)
ts = dd.timeseries(
    mode="mean",
    f_range=(f_center - 8 * dd.df, f_center + 8 * dd.df),
)

fig, ax = plt.subplots(figsize=(8, 3))
ts.plot(ttype="trel")
ax.set_title("Mean power in a narrow band around the dedrifted peak")
display(fig)
plt.close(fig)

print("time series shape:", ts.shape)
print("time series metadata:", ts.metadata["derived"])

In [ ]:
one_d_spectrum = stg.Spectrum(df=dd.df, dt=dd.obs_length, fch1=dd.fch1, data=spectrum.array())
one_d_timeseries = stg.TimeSeries(df=ts.df, dt=ts.dt, fch1=ts.fch1, data=ts.array())
print("1D spectrum coerced shape:", one_d_spectrum.shape)
print("1D time series coerced shape:", one_d_timeseries.shape)

In [ ]:
h5_path = OUT / "derived_product_source.h5"
frame.save_hdf5(h5_path)
with stg.Frame.open(h5_path, mode="r", max_chunk_bytes=4096) as backed:
    backed_spectrum = backed.spectrum(
        mode="sum",
        f_index_range=(350, 500),
        max_chunk_bytes=4096,
    )
    backed_ts = backed.timeseries(
        mode="mean",
        f_index_range=(350, 500),
        max_chunk_bytes=4096,
    )

print("file-backed spectrum:", backed_spectrum.shape, backed_spectrum.metadata["derived"])
print("file-backed time series:", backed_ts.shape, backed_ts.metadata["derived"])